In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :exponential

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model] Fitting chain 2 (tau=59)
[ Info: [exponential] iter 1000/1000000 elapsed=5.4s, rate=0.054, mean=[0.918, 0.00047, 0.372, 0.400], std=[0.0234, 0.000450, 0.0320, 0.0193] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=9.9s, rate=0.072, mean=[0.842, 0.00072, 0.463, 0.337], std=[0.0838, 0.000460, 0.0884, 0.0691] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=13.7s, rate=0.078, mean=[0.800, 0.00119, 0.492, 0.257], std=[0.0868, 0.000714, 0.0816, 0.1174] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=17.4s, rate=0.078, mean=[0.782, 0.00145, 0.505, 0.213], std=[0.0801, 0.000739, 0.0736, 0.1223] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=21.1s, rate=0.080, mean=[0.766, 0.00170, 0.516, 0.184], std=[0.0772, 0.000802, 0.0688, 0.1207] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=24.9s, rate=0.078, mean=[0.753, 0.00189, 0.522, 0.165], std=[0.0757, 0.000831, 0.0638, 0.1170] [ADAPT]
[ Info: [exponential] iter 7000/1000000 ela